# Main coverage-test suite (Exp 1/2) + BMA-vs-single comparison + rotated_memory_z circuit spot-check + LOO model comparison

In [ ]:
!pip install pymc3 arviz stim pymatching

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import numpy as np
import pymc as pm
import arviz as az
from scipy import stats as sstats
from scipy.special import logsumexp, softmax
import math
import warnings
warnings.filterwarnings('ignore')

RNG_SEED = 12345
np.random.seed(RNG_SEED)

SAMPLE_CORES = 1

print("Setup complete.")

E_IDEAL = 1.0 / np.sqrt(2)
COEFF_NET = E_IDEAL

A_TRUE = E_IDEAL
B_TRUE = -0.053
C_TRUE = 0.45

B1_TRUE, C1_TRUE = -0.06, 0.50
B2_TRUE, C2_TRUE = -0.015, 0.12

DISTANCES = np.array([5, 7, 9, 11, 13], dtype=float)

def true_PL_single(d):
    E = A_TRUE + B_TRUE * np.exp(-C_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def true_PL_double(d):
    E = A_TRUE + B1_TRUE * np.exp(-C1_TRUE * d) + B2_TRUE * np.exp(-C2_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def evaluate_tstate_properties(PL_x, n_shots):

    E_T = COEFF_NET * (1.0 - 2.0 * PL_x)
    var_pm = (COEFF_NET**2) * (4.0 * PL_x * (1.0 - PL_x) / n_shots)
    return E_T, np.sqrt(var_pm)

def generate_synthetic_data(distances, n_shots, truth_type='single', rng=None):
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    E_obs, sigma_obs = [], []
    for d in distances:
        pl_true = true_PL_single(d) if truth_type == 'single' else true_PL_double(d)
        n_errors = rng.binomial(n_shots, pl_true)
        pl_obs = n_errors / n_shots
        e_t, sig = evaluate_tstate_properties(pl_obs, n_shots)
        E_obs.append(e_t)
        sigma_obs.append(sig)
    return np.array(E_obs), np.array(sigma_obs)

print("Ground truths loaded.")

def build_single_exp_model(d_obs, E_obs, sigma_obs):
    with pm.Model() as model:
        A = pm.Uniform("A", lower=0.65, upper=0.75)
        log_neg_B = pm.Normal("log_neg_B", mu=-1.0, sigma=1.5)
        B = pm.Deterministic("B", -pm.math.exp(log_neg_B))
        log_C = pm.Normal("log_C", mu=0.0, sigma=1.0)
        C = pm.Deterministic("C", pm.math.exp(log_C))
        E_pred = A + B * pm.math.exp(-C * d_obs)
        pm.Normal("obs", mu=E_pred, sigma=sigma_obs, observed=E_obs)
    return model

def build_double_exp_model(d_obs, E_obs, sigma_obs):
    with pm.Model() as model:
        A = pm.Uniform("A", lower=0.65, upper=0.75)
        log_neg_B1 = pm.Normal("log_neg_B1", mu=-1.5, sigma=1.0)
        B1 = pm.Deterministic("B1", -pm.math.exp(log_neg_B1))
        log_C1 = pm.Normal("log_C1", mu=0.3, sigma=0.7)
        C1 = pm.Deterministic("C1", pm.math.exp(log_C1))
        log_neg_B2 = pm.Normal("log_neg_B2", mu=-3.0, sigma=1.0)
        B2 = pm.Deterministic("B2", -pm.math.exp(log_neg_B2))
        frac = pm.Beta("frac_C2_of_C1", alpha=2, beta=5)
        C2 = pm.Deterministic("C2", C1 * frac)
        E_pred = A + B1 * pm.math.exp(-C1 * d_obs) + B2 * pm.math.exp(-C2 * d_obs)
        pm.Normal("obs", mu=E_pred, sigma=sigma_obs, observed=E_obs)
    return model

def fit_with_diagnostics(model, draws, tune, target_accept=0.95):
    """Runs NUTS and returns (trace, diagnostics_dict) instead of just a
    trace, so bad fits are visible instead of silently trusted."""
    with model:
        trace = pm.sample(draws=draws, tune=tune, chains=2,
                           target_accept=target_accept, cores=SAMPLE_CORES,
                           random_seed=RNG_SEED, progressbar=False)
    var_names = [v for v in ["A", "B", "C", "B1", "C1", "B2", "C2"]
                 if v in trace.posterior]
    summary = az.summary(trace, var_names=var_names)
    diag = {
        "divergences": int(trace.sample_stats.diverging.sum()),
        "min_ess_bulk": float(summary["ess_bulk"].min()),
        "max_rhat": float(summary["r_hat"].max()),
    }
    diag["clean"] = (diag["divergences"] == 0 and diag["min_ess_bulk"] >= 400
                      and diag["max_rhat"] <= 1.01)
    return trace, diag

print("Model builders ready.")


def prior_only_baseline(model_builder, d_obs, prob=0.95, draws=4000):
    """Samples A from the prior alone (no likelihood) and checks whether
    the resulting interval already contains A_TRUE."""
    dummy_E = np.zeros(len(d_obs))
    dummy_sigma = np.ones(len(d_obs))
    model = model_builder(d_obs, dummy_E, dummy_sigma)
    with model:
        prior = pm.sample_prior_predictive(draws=draws, random_seed=RNG_SEED)
    A_prior = prior.prior["A"].values.flatten()

    hdi = az.hdi(A_prior, prob=prob)
    contains = bool(hdi[0] <= A_TRUE <= hdi[1])
    return {"hdi_low": float(hdi[0]), "hdi_high": float(hdi[1]), "contains_A_TRUE": contains}

baseline_single = prior_only_baseline(build_single_exp_model, DISTANCES)
print("Prior-only baseline (single-exp model):", baseline_single)


def run_coverage_test(n_trials, n_shots, truth_type, draws, tune, target_accept=0.95):
    rng = np.random.default_rng(RNG_SEED + 9999)
    results, diagnostics = [], []

    for i in range(n_trials):
        E_obs, sigma_obs = generate_synthetic_data(DISTANCES, n_shots, truth_type, rng)

        model = build_single_exp_model(DISTANCES, E_obs, sigma_obs)

        trace, diag = fit_with_diagnostics(model, draws=draws, tune=tune,
                                            target_accept=target_accept)
        A_samples = trace.posterior["A"].values.flatten()
        hdi = az.hdi(A_samples, prob=0.95)
        contains = bool(hdi[0] <= A_TRUE <= hdi[1])

        results.append(contains)
        diagnostics.append(diag)

        if (i + 1) % 10 == 0:
            n_dirty = sum(not d["clean"] for d in diagnostics)
            print(f"  Trial {i+1}/{n_trials} done. ({n_dirty} trials flagged unclean so far)")

    coverage = np.mean(results)
    n_success = int(sum(results))
    ci_low, ci_high = sstats.beta.ppf([0.025, 0.975], n_success + 1, n_trials - n_success + 1)
    n_dirty = sum(not d["clean"] for d in diagnostics)

    print(f"\n--- Coverage Test ({truth_type} truth, {n_shots} shots) ---")
    print(f"Coverage: {coverage*100:.1f}% ({n_success}/{n_trials})")
    print(f"95% Bayesian credible interval on coverage: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
    print(f"Trials flagged with sampling problems (divergences / low ESS / high rhat): {n_dirty}/{n_trials}")
    if n_dirty > 0:
        print("Coverage recomputed excluding flagged trials:",
              f"{np.mean([r for r, d in zip(results, diagnostics) if d['clean']])*100:.1f}%")

    return results, diagnostics


Setup complete.
Ground truths loaded.
Model builders ready.
Prior-only baseline (single-exp model): {'hdi_low': 0.6562481537912734, 'hdi_high': 0.7497392830036317, 'contains_A_TRUE': True}


In [ ]:
res_correct, diag_correct = run_coverage_test(
    n_trials=150, n_shots=1_000_000, truth_type='single', draws=2000, tune=1500)
res_misspec, diag_misspec = run_coverage_test(
    n_trials=150, n_shots=1_000_000, truth_type='double', draws=2000, tune=1500)

  Trial 10/150 done. (0 trials flagged unclean so far)
  Trial 20/150 done. (0 trials flagged unclean so far)
  Trial 30/150 done. (0 trials flagged unclean so far)
  Trial 40/150 done. (0 trials flagged unclean so far)
  Trial 50/150 done. (0 trials flagged unclean so far)
  Trial 60/150 done. (0 trials flagged unclean so far)
  Trial 70/150 done. (0 trials flagged unclean so far)
  Trial 80/150 done. (0 trials flagged unclean so far)
  Trial 90/150 done. (0 trials flagged unclean so far)
  Trial 100/150 done. (0 trials flagged unclean so far)
  Trial 110/150 done. (0 trials flagged unclean so far)
  Trial 120/150 done. (0 trials flagged unclean so far)
  Trial 130/150 done. (0 trials flagged unclean so far)
  Trial 140/150 done. (0 trials flagged unclean so far)
  Trial 150/150 done. (0 trials flagged unclean so far)

--- Coverage Test (single truth, 1000000 shots) ---
Coverage: 94.7% (142/150)
95% Bayesian credible interval on coverage: [89.8%, 97.2%]
Trials flagged with sampling pr

In [ ]:
rng_check = np.random.default_rng(RNG_SEED + 555)
E_chk, sigma_chk = generate_synthetic_data(DISTANCES, 1_000_000, 'double', rng_check)
model_chk = build_double_exp_model(DISTANCES, E_chk, sigma_chk)
trace_chk, diag_chk = fit_with_diagnostics(model_chk, draws=500, tune=800, target_accept=0.97)
print("Double-exp model, correctly specified, geometry check:", diag_chk)

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Double-exp model, correctly specified, geometry check: {'divergences': 0, 'min_ess_bulk': 168.0, 'max_rhat': 1.0, 'clean': False}


In [ ]:
!pip install stim pymatching

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.9/625.9 kB 32.2 MB/s eta 0:00:00


In [ ]:
import stim
import pymatching

def logical_error_rate(distance, rounds, p, shots, seed):
    """STAND-IN circuit. Replace with your actual injection/measurement
    circuit before trusting this cell's output."""
    circuit = stim.Circuit.generated(
        'surface_code:rotated_memory_z',
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler(seed=seed)
    det, obs = sampler.sample(shots=shots, separate_observables=True)
    pred = matcher.decode_batch(det)
    n_errors = int(np.sum(pred[:, 0] != obs[:, 0]))
    return n_errors / shots

def validate_variance_formula(distance, p, n_shots, repeats, rounds=None):
    """Runs the circuit `repeats` independent times at fixed distance,
    computes the empirical variance of E_T across those repeats, and
    compares it to the theoretical formula evaluated at the mean PL."""
    if rounds is None:
        rounds = int(distance)
    E_vals = []
    for r in range(repeats):
        pl = logical_error_rate(distance, rounds, p, n_shots, seed=RNG_SEED + r)
        e_t, _ = evaluate_tstate_properties(pl, n_shots)
        E_vals.append(e_t)
    E_vals = np.array(E_vals)
    empirical_var = np.var(E_vals, ddof=1)
    mean_pl = np.mean([(1 - e / COEFF_NET) / 2 for e in E_vals])
    _, theo_sigma = evaluate_tstate_properties(mean_pl, n_shots)
    theo_var = theo_sigma ** 2
    return {
        "distance": distance, "mean_PL": mean_pl,
        "empirical_var": empirical_var, "theoretical_var": theo_var,
        "ratio": empirical_var / theo_var,
    }

for d in [5, 9, 13]:
    result = validate_variance_formula(distance=d, p=0.005, n_shots=5000, repeats=30)
    print(result)

{'distance': 5, 'mean_PL': np.float64(0.013726666666666668), 'empirical_var': np.float64(4.6923218390805e-06), 'theoretical_var': np.float64(5.415298115555557e-06), 'ratio': np.float64(0.8664937255442517)}
{'distance': 9, 'mean_PL': np.float64(0.00667999999999999), 'empirical_var': np.float64(3.5398620689655322e-06), 'theoretical_var': np.float64(2.654151039999995e-06), 'ratio': np.float64(1.3337078469225092)}
{'distance': 13, 'mean_PL': np.float64(0.002773333333333326), 'empirical_var': np.float64(8.702528735632061e-07), 'theoretical_var': np.float64(1.1062567822222193e-06), 'ratio': np.float64(0.7866644413379913)}


In [ ]:

def compute_exact_loo(d_arr, E_obs, sigma_obs, model_builder, model_name):
    n = len(d_arr)
    log_preds = []
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        d_train, E_train, sig_train = d_arr[mask], E_obs[mask], sigma_obs[mask]

        model = model_builder(d_train, E_train, sig_train)
        trace, diag = fit_with_diagnostics(model, draws=2500, tune=2000, target_accept=0.95)
        if not diag["clean"]:
            print(f"  WARNING: {model_name} fold {i+1}/{n} did not sample cleanly: {diag}")

        A_s = trace.posterior["A"].values.flatten()
        if "B" in trace.posterior:
            B_s = trace.posterior["B"].values.flatten()
            C_s = trace.posterior["C"].values.flatten()
            E_pred_s = A_s + B_s * np.exp(-C_s * d_arr[i])
        else:
            B1_s = trace.posterior["B1"].values.flatten()
            C1_s = trace.posterior["C1"].values.flatten()
            B2_s = trace.posterior["B2"].values.flatten()
            C2_s = trace.posterior["C2"].values.flatten()
            E_pred_s = A_s + B1_s * np.exp(-C1_s * d_arr[i]) + B2_s * np.exp(-C2_s * d_arr[i])

        log_liks = sstats.norm.logpdf(E_obs[i], loc=E_pred_s, scale=sigma_obs[i])
        log_pred = logsumexp(log_liks) - np.log(len(log_liks))
        log_preds.append(log_pred)
        print(f"  {model_name} Fold {i+1}/{n}: log_pred={log_pred:.4f}")

    return float(np.sum(log_preds))

def compare_models(d_arr, E_obs, sigma_obs):
    loo_single = compute_exact_loo(d_arr, E_obs, sigma_obs, build_single_exp_model, "Single")
    loo_double = compute_exact_loo(d_arr, E_obs, sigma_obs, build_double_exp_model, "Double")

    # Primary result: raw exact LOO, no extra penalty.
    weights_raw = softmax([loo_single, loo_double])
    print("\n--- Model Comparison (PRIMARY: raw exact LOO) ---")
    print(f"Raw LOO Single: {loo_single:.4f}, Raw LOO Double: {loo_double:.4f}")
    print(f"Model Weights: Single={weights_raw[0]:.3f}, Double={weights_raw[1]:.3f}")

    # Rejected alternative, kept for transparency.
    n = len(d_arr)
    penalty_single, penalty_double = (3/2)*math.log(n), (5/2)*math.log(n)
    weights_pen = softmax([loo_single - penalty_single, loo_double - penalty_double])
    print("\n--- Comparison (REJECTED: BIC-penalty stacked on exact LOO) ---")
    print(f"Penalized weights: Single={weights_pen[0]:.3f}, Double={weights_pen[1]:.3f}")
    print("Rejected because: exact LOO already penalizes overfitting via held-out")
    print("prediction; adding a second (k/2)log(n) penalty double-counts complexity")
    print("cost, and here the penalty gap (%.3f) exceeds the raw predictive gap (%.3f)."
          % (penalty_double - penalty_single, loo_double - loo_single))

    return {"loo_single": loo_single, "loo_double": loo_double,
            "weights_raw": weights_raw.tolist(), "weights_penalized": weights_pen.tolist()}

In [ ]:
rng_loo = np.random.default_rng(RNG_SEED + 777)
E_obs_loo, sigma_obs_loo = generate_synthetic_data(DISTANCES, 1_000_000, 'double', rng_loo)
comparison_result = compare_models(DISTANCES, E_obs_loo, sigma_obs_loo)

  Single Fold 1/5: log_pred=5.7613
  Single Fold 2/5: log_pred=7.1036
  Single Fold 3/5: log_pred=7.8796
  Single Fold 4/5: log_pred=7.8999
  Single Fold 5/5: log_pred=6.7546
  Double Fold 1/5: log_pred=6.0948
  Double Fold 2/5: log_pred=7.4513


ERROR:pymc.stats.convergence:There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


  Double Fold 3/5: log_pred=8.0027
  Double Fold 4/5: log_pred=8.1433


ERROR:pymc.stats.convergence:There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


  Double Fold 5/5: log_pred=7.4075

--- Model Comparison (PRIMARY: raw exact LOO) ---
Raw LOO Single: 35.3990, Raw LOO Double: 37.0996
Model Weights: Single=0.154, Double=0.846

--- Comparison (REJECTED: BIC-penalty stacked on exact LOO) ---
Penalized weights: Single=0.477, Double=0.523
Rejected because: exact LOO already penalizes overfitting via held-out
prediction; adding a second (k/2)log(n) penalty double-counts complexity
cost, and here the penalty gap (1.609) exceeds the raw predictive gap (1.701).


In [ ]:

def true_PL_double_scaled(d, severity):
    """Same as true_PL_double, but B2's amplitude is scaled by `severity`.
    severity=1 reproduces your original misspecification exactly."""
    B2 = B2_TRUE * severity
    E = A_TRUE + B1_TRUE * np.exp(-C1_TRUE * d) + B2 * np.exp(-C2_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def generate_synthetic_data_scaled(distances, n_shots, severity, rng):
    E_obs, sigma_obs = [], []
    for d in distances:
        pl_true = true_PL_double_scaled(d, severity)
        n_errors = rng.binomial(n_shots, pl_true)
        pl_obs = n_errors / n_shots
        e_t, sig = evaluate_tstate_properties(pl_obs, n_shots)
        E_obs.append(e_t)
        sigma_obs.append(sig)
    return np.array(E_obs), np.array(sigma_obs)

def severity_sweep_coverage(severity, n_trials, n_shots, draws=2000, tune=1500,
                             target_accept=0.98):
    """Same as run_coverage_test, but also tracks bias (posterior mean - truth)
    and interval width per trial, not just the coverage boolean."""
    rng = np.random.default_rng(RNG_SEED + 9999)
    biases, widths, covers, diagnostics = [], [], [], []
    n_failed = 0

    for i in range(n_trials):
        try:
            E_obs, sigma_obs = generate_synthetic_data_scaled(DISTANCES, n_shots, severity, rng)
            model = build_single_exp_model(DISTANCES, E_obs, sigma_obs)
            trace, diag = fit_with_diagnostics(model, draws=draws, tune=tune,
                                                target_accept=target_accept)
            A_s = trace.posterior["A"].values.flatten()
            hdi = az.hdi(A_s, prob=0.95)
            biases.append(float(np.mean(A_s)) - A_TRUE)
            widths.append(float(hdi[1] - hdi[0]))
            covers.append(bool(hdi[0] <= A_TRUE <= hdi[1]))
            diagnostics.append(diag)
        except Exception as e:

            n_failed += 1
            print(f"  [severity={severity}] trial {i+1}/{n_trials} FAILED: "
                  f"{type(e).__name__}: {e}")
            continue

        if (i + 1) % 10 == 0:
            print(f"  [severity={severity}] {i+1}/{n_trials} done "
                  f"({n_failed} failed so far)")

    n_ok = len(covers)
    n_dirty = sum(not d["clean"] for d in diagnostics)
    result = {
        "severity": severity,
        "coverage": float(np.mean(covers)) if n_ok else None,
        "mean_bias": float(np.mean(biases)) if n_ok else None,
        "bias_std": float(np.std(biases)) if n_ok else None,
        "mean_interval_width": float(np.mean(widths)) if n_ok else None,
        "n_flagged": n_dirty,
        "n_failed": n_failed,
        "n_trials": n_trials,
    }
    print(result)
    return result

In [ ]:
sweep_results = []
for severity in [1, 3, 6, 10, 20, 40, 80]:
    r = severity_sweep_coverage(severity, n_trials=30, n_shots=1_000_000)
    sweep_results.append(r)

  [severity=1] 10/30 done (0 failed so far)
  [severity=1] 20/30 done (0 failed so far)
  [severity=1] 30/30 done (0 failed so far)
{'severity': 1, 'coverage': 0.0, 'mean_bias': -0.002051066444096624, 'bias_std': 0.00017390260901273434, 'mean_interval_width': 0.0006719169905242979, 'n_flagged': 0, 'n_failed': 0, 'n_trials': 30}
  [severity=3] 10/30 done (0 failed so far)
  [severity=3] 20/30 done (0 failed so far)
  [severity=3] 30/30 done (0 failed so far)
{'severity': 3, 'coverage': 0.0, 'mean_bias': -0.004092123144938864, 'bias_std': 0.0005856380111121006, 'mean_interval_width': 0.002110730618174195, 'n_flagged': 2, 'n_failed': 0, 'n_trials': 30}
  [severity=6] 10/30 done (0 failed so far)
  [severity=6] 20/30 done (0 failed so far)
  [severity=6] 30/30 done (0 failed so far)
{'severity': 6, 'coverage': 0.0, 'mean_bias': -0.005484775208857166, 'bias_std': 0.0011060469688045171, 'mean_interval_width': 0.0041452394741534355, 'n_flagged': 0, 'n_failed': 0, 'n_trials': 30}
  [severity=1

In [ ]:
def fit_full(model, draws=2000, tune=1000, target_accept=0.95):
    with model:
        return pm.sample(draws=draws, tune=tune, chains=2,
                          target_accept=target_accept, cores=SAMPLE_CORES,
                          random_seed=RNG_SEED, progressbar=False)

def bma_vs_single_coverage(severity, n_trials, n_shots,
                            loo_draws=500, loo_tune=500,
                            full_draws=2000, full_tune=1000):
    rng = np.random.default_rng(RNG_SEED + 9999)
    single_covers, bma_covers, bma_weights_double = [], [], []

    for i in range(n_trials):
        E_obs, sigma_obs = generate_synthetic_data_scaled(DISTANCES, n_shots, severity, rng)

        # Exact LOO scores for both models (5 refits each, same method as Cell 7)
        loo_single = compute_exact_loo(DISTANCES, E_obs, sigma_obs,
                                        build_single_exp_model, "Single")
        loo_double = compute_exact_loo(DISTANCES, E_obs, sigma_obs,
                                        build_double_exp_model, "Double")
        weights = softmax([loo_single, loo_double])

        # Full-data posteriors, for the actual pooled coverage check
        trace_s, _ = fit_with_diagnostics(build_single_exp_model(DISTANCES, E_obs, sigma_obs),
                                           draws=full_draws, tune=full_tune)
        trace_d, _ = fit_with_diagnostics(build_double_exp_model(DISTANCES, E_obs, sigma_obs),
                                           draws=full_draws, tune=full_tune)
        A_single = trace_s.posterior["A"].values.flatten()
        A_double = trace_d.posterior["A"].values.flatten()

        # Pool posterior samples in proportion to model weight
        n_pool = 4000
        n_from_single = int(round(weights[0] * n_pool))
        n_from_double = n_pool - n_from_single
        pooled = np.concatenate([
            np.random.default_rng(i * 2).choice(A_single, n_from_single, replace=True),
            np.random.default_rng(i * 2 + 1).choice(A_double, n_from_double, replace=True),
        ])

        hdi_single = az.hdi(A_single, prob=0.95)
        hdi_bma = az.hdi(pooled, prob=0.95)
        single_covers.append(bool(hdi_single[0] <= A_TRUE <= hdi_single[1]))
        bma_covers.append(bool(hdi_bma[0] <= A_TRUE <= hdi_bma[1]))
        bma_weights_double.append(float(weights[1]))

        print(f"  [severity={severity}] trial {i+1}/{n_trials}: "
              f"single_covers={single_covers[-1]}, bma_covers={bma_covers[-1]}, "
              f"weight_double={weights[1]:.3f}")

    result = {
        "severity": severity,
        "single_coverage": float(np.mean(single_covers)),
        "bma_coverage": float(np.mean(bma_covers)),
        "mean_weight_on_double": float(np.mean(bma_weights_double)),
        "n_trials": n_trials,
    }
    print(result)
    return result


In [ ]:
bma_results = []
for severity in [1, 20, 80]:
    r = bma_vs_single_coverage(severity, n_trials=15, n_shots=1_000_000)
    bma_results.append(r)

  Single Fold 1/5: log_pred=6.5984
  Single Fold 2/5: log_pred=7.8802
  Single Fold 3/5: log_pred=7.6039
  Single Fold 4/5: log_pred=7.3246
  Single Fold 5/5: log_pred=7.1404
  Double Fold 1/5: log_pred=5.9391


ERROR:pymc.stats.convergence:There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


  Double Fold 2/5: log_pred=7.1011
  Double Fold 3/5: log_pred=7.1940
  Double Fold 4/5: log_pred=7.5907
  Double Fold 5/5: log_pred=7.3813
  [severity=1] trial 1/15: single_covers=False, bma_covers=False, weight_double=0.207
  Single Fold 1/5: log_pred=6.4131
  Single Fold 2/5: log_pred=7.8841
  Single Fold 3/5: log_pred=7.4293
  Single Fold 4/5: log_pred=6.4295
  Single Fold 5/5: log_pred=6.3483


ERROR:pymc.stats.convergence:There were 7 divergences after tuning. Increase `target_accept` or reparameterize.


  Double Fold 1/5: log_pred=5.9046


...

  Double Fold 5/5: log_pred=5.6167


ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


  [severity=20] trial 5/15: single_covers=False, bma_covers=False, weight_double=0.125
  Single Fold 1/5: log_pred=5.5650
  Single Fold 2/5: log_pred=6.4479
  Single Fold 3/5: log_pred=5.8363
  Single Fold 4/5: log_pred=6.4998
  Single Fold 5/5: log_pred=6.0502


ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


  Double Fold 1/5: log_pred=5.2949


In [ ]:
bma_results = []
for severity in [20, 80]:
    if severity == 20:
      r = bma_vs_single_coverage(severity, n_trials=10, n_shots=1_000_000)
    else:
      r = bma_vs_single_coverage(severity, n_trials=15, n_shots=1_000_000)
    bma_results.append(r)